## Load Model

In [ ]:
from unsloth import FastModel
import torch

In [ ]:
max_seq_length = 1024
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # bnb quantization is CPU-bottlenecked on Blackwell
    load_in_8bit = False,   # same issue - skip bnb entirely
    full_finetuning = False,
    fast_inference = True,  # enables vLLM-style fast inference
    # token = "YOUR_HF_TOKEN",
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)


## Load Data

In [ ]:
from datasets import load_dataset
dataset = load_dataset("openai/gsm8k", "main", split = "train")
dataset

In [ ]:
def extract_hash_answer(text):
    if "####" not in text: return None
    return text.split("####")[1].strip()
extract_hash_answer(dataset[0]["answer"])

In [ ]:
reasoning_start = "<think>"
reasoning_end   = "</think>"
solution_start = "<answer>"
solution_end = "</answer>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["question"]},
    ],
    "answer": extract_hash_answer(x["answer"]),
})
dataset[0]

In [ ]:
import re

match_format = re.compile(
    rf"^[\s]{{0,}}"\
    rf"{reasoning_start}.+?{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end)   == 1 else -0.5
        score += 0.5 if response.count(solution_start)  == 1 else -0.5
        score += 0.5 if response.count(solution_end)    == 1 else -0.5
        scores.append(score)
    return scores

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_torch_fused",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,  # Accumulate 4 steps for smoother training
    num_generations = 8,              # 8 generations per prompt (batched via vLLM → fast)
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1,
    max_steps = 50,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    # ── GRPO-specific perf settings ──
    bf16 = True,                      # BF16 training on Blackwell
    loss_type = "dapo",               # DAPO loss (removes length bias, default in latest unsloth)
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

In [ ]:
import time
from vllm import SamplingParams

NUM_TOKENS = 256
prompt = "Write a super duper long story about a dragon"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)

sampling_params = SamplingParams(
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_tokens=NUM_TOKENS,
)

# Save + load LoRA adapter for vLLM fast path
lora_dir = "/tmp/unsloth_lora_bench"
model.save_lora(lora_dir)
lora_req = model.load_lora(lora_dir)

# ── Warmup ──
_ = model.fast_generate([text], sampling_params=sampling_params, lora_request=lora_req)

# ── Benchmark fast_generate (vLLM) ──
times = []
for i in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=lora_req,
    )
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    gen_tokens = len(out[0].outputs[0].token_ids)
    times.append((t1 - t0, gen_tokens))

for i, (elapsed, ntok) in enumerate(times):
    print(f"  Run {i}: {elapsed:.3f}s | {ntok} tokens | {ntok/elapsed:.1f} tok/s")
avg_time = sum(t for t, _ in times) / len(times)
avg_tok = sum(n for _, n in times) / len(times)
print(f"\n  fast_generate avg: {avg_time:.3f}s | {avg_tok:.0f} tokens | {avg_tok/avg_time:.1f} tok/s")
print(f"\n  NOTE: model.generate() was ~10 tok/s (HuggingFace path)")
print(f"  model.fast_generate() uses vLLM backend → {avg_tok/avg_time:.0f} tok/s")
print(f"  GRPOTrainer automatically uses fast_generate when fast_inference=True")

In [ ]:
# ── Batched throughput benchmark (GRPO generates multiple completions per prompt) ──
batch_sizes = [1, 4, 8, 16]
NUM_TOKENS = 512

sampling_params = SamplingParams(
    temperature=1.0, top_p=0.95, top_k=64, max_tokens=NUM_TOKENS,
)

# load_tensors=True → extracts LoRA directly from model state (no disk I/O)
lora_req_fast = model.load_lora(lora_dir, load_tensors=True)

print(f"{'Batch':>5} | {'Time':>7} | {'Tokens':>8} | {'Throughput':>12} | {'Per-seq':>10}")
print("-" * 60)
for bs in batch_sizes:
    prompts = [text] * bs
    _ = model.fast_generate(prompts, sampling_params=sampling_params, lora_request=lora_req_fast)  # warmup
    
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.fast_generate(prompts, sampling_params=sampling_params, lora_request=lora_req_fast)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    
    total_tokens = sum(len(o.outputs[0].token_ids) for o in out)
    print(f"{bs:>5} | {elapsed:>6.3f}s | {total_tokens:>8} | {total_tokens/elapsed:>10.1f}/s | {total_tokens/elapsed/bs:>8.1f}/s")